# Final Submission Pipeline: Phase 46

## Pipeline Sequence
* **Load Data:** Imports the train and test sets strictly from the allowed competition path.
* **Dynamic Bounds:** Calculates optimal clipping boundaries from the training set by mathematically forcing a normal target distribution (Kurtosis = 3.0).
* **Feature Engineering:** Computes simple autoregressive features (Lags 1, 2, and 3).
* **Winsorization:** Applies robust clipping to all features and asymmetric clipping to the target to handle extreme outliers.
* **Base Model:** Trains the core regression algorithm.
* **Modulator Extraction:** Computes three non-linear feature interactions statelessly.
* **Alpha Application:** Applies our offline-tuned weights to adjust the base predictions sequentially.
* **Post-Processing:** Scales only the positive tail of predictions to match the true target amplitude.
* **Final Export:** Clips the adjusted predictions using the dynamic bounds from Phase 1 and saves the final file.

## Core Model
* **PLS Regression:** Used a 2-component Partial Least Squares model. It provides a highly stable, beta-neutral foundation.

## Modulators
* **Mod 1:** A linear interaction combining specific S03 signals with price lags.
* **Mod 2:** A 2-D stack using the energy product of ReLU-activated features.
* **Mod 3 (Omega Stack):** A 5-dimensional non-linear stack capturing complex market regime shifts.

## Static Hyperparameters
* **Feature Winsorization:** 0.1% to 99.9%.
* **Target Winsorization:** 1.19% (lower) and 98.8% (upper).
* **Modulator Multipliers (C):** -0.46857, -0.15092, and 0.06037.
* **Positive Tail Scalar:** 1.1x.

## Why I Chose This Approach
During validation, ensembling and bagging smoothed our predictions too much, which the R-squared evaluation metric actively punishes. This highly regularized, monolithic approach preserves the necessary prediction variance. By applying a strict 1.1x scalar to the positive tail, we achieve the exact amplitude required to score well, while our dynamic Kurtosis clipping acts as an absolute shield against black swan events.

In [1]:
import pandas as pd
import numpy as np
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
import scipy.stats as stats
from scipy.optimize import minimize_scalar
import warnings
warnings.filterwarnings('ignore')

print("Executing Production Pipeline...")

# NOTE: 
# The CFG class contains static hyperparameters (percentiles, modulator weights, and scalars) 
class CFG:
    feat_winsor_p = 0.001       
    target_lower_p = 0.011916
    target_upper_p = 0.988494
    mod1_C = -0.46857           
    mod2_C = -0.15092           
    mod3_C = 0.06037            
    asym_positive_scale = 1.1   

train_df = pd.read_parquet("/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/train.parquet")
test_df = pd.read_parquet("/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/test.parquet")

# NOTE:
# Target clipping bounds are solved dynamically end-to-end using the training distribution's 
# Median Absolute Deviation to force a normal kurtosis of 3.0. 
target = train_df['TARGET'].dropna()
median_val = np.median(target)
mad_val = stats.median_abs_deviation(target)

def kurtosis_objective(m):
    clipped = np.clip(target, median_val - (m * mad_val), median_val + (m * mad_val))
    return (pd.Series(clipped).kurtosis() - 3.0)**2

res = minimize_scalar(kurtosis_objective, bounds=(1, 15), method='bounded')
optimal_mad_multiplier = res.x

lower_bound_val = median_val - (optimal_mad_multiplier * mad_val)
upper_bound_val = median_val + (optimal_mad_multiplier * mad_val)

# NOTE:
# Autoregressive features are generated using standard row-wise mathematics.
safe_price_train = train_df['Price'] + 1e-6
train_df['Past_Ret_1'] = 100 * (train_df['Price_LagT1'] / safe_price_train)
train_df['Past_Ret_2'] = 100 * (train_df['Price_LagT2'] / safe_price_train)
train_df['Past_Ret_3'] = 100 * (train_df['Price_LagT3'] / safe_price_train)

safe_price_test = test_df['Price'] + 1e-6
test_df['Past_Ret_1'] = 100 * (test_df['Price_LagT1'] / safe_price_test)
test_df['Past_Ret_2'] = 100 * (test_df['Price_LagT2'] / safe_price_test)
test_df['Past_Ret_3'] = 100 * (test_df['Price_LagT3'] / safe_price_test)

ar_features = ['Past_Ret_1', 'Past_Ret_2', 'Past_Ret_3']

df_train_clean = train_df.dropna(subset=ar_features).copy()
mask_clean = (df_train_clean['Past_Ret_1'].abs() < 50) & (df_train_clean['Past_Ret_2'].abs() < 50) & (df_train_clean['Past_Ret_3'].abs() < 50)
df_train_clean = df_train_clean[mask_clean]

mask_safe = test_df['Past_Ret_1'].notna() & (test_df['Past_Ret_1'].abs() < 50) & (test_df['Past_Ret_2'].abs() < 50) & (test_df['Past_Ret_3'].abs() < 50)

# NOTE:
# Feature winsorization relies strictly on aggregate percentiles derived from the training set,
# which are applied globally and statelessly to the test set.
for col in ar_features:
    lower_bound_f = df_train_clean[col].quantile(CFG.feat_winsor_p)
    upper_bound_f = df_train_clean[col].quantile(1.0 - CFG.feat_winsor_p)
    
    df_train_clean[col] = np.clip(df_train_clean[col], lower_bound_f, upper_bound_f)
    test_df.loc[mask_safe, col] = np.clip(test_df.loc[mask_safe, col], lower_bound_f, upper_bound_f)

final_lower = df_train_clean['TARGET'].quantile(CFG.target_lower_p)
final_upper = df_train_clean['TARGET'].quantile(CFG.target_upper_p)

df_train_clean['TARGET_WINSORIZED'] = np.clip(df_train_clean['TARGET'], final_lower, final_upper)
df_train_clean['TARGET_NEUTRAL'] = df_train_clean.groupby('CV_GROUP')['TARGET_WINSORIZED'].transform(lambda x: x - x.mean())

y_ar_train_neutral = df_train_clean['TARGET_NEUTRAL'].values

scaler = StandardScaler()
X_ar_train = df_train_clean[ar_features].values
X_test_safe = test_df.loc[mask_safe, ar_features].values

X_train_scaled = scaler.fit_transform(X_ar_train)
X_test_scaled = scaler.transform(X_test_safe)

pls_model = PLSRegression(n_components=2, scale=False)
pls_model.fit(X_train_scaled, y_ar_train_neutral)

base_test_preds = pls_model.predict(X_test_scaled).ravel()

def standardize(vec): return (vec - np.mean(vec)) / (np.std(vec) + 1e-8)
def relu(x): return np.maximum(0, x)
def energy_product(x, y): return 0.5 * (x**2) * (y**2)
def sign_sq(x): return np.sign(x) * (x**2)
def sub(x, y): return x - y
def max_pool(x, y): return np.maximum(x, y)
def interaction_abs(x, y): return x * np.abs(y)

grail_vec_tr = df_train_clean['S03_A07_A05_V09_LagT1'].fillna(0).values + df_train_clean['Price_LagT3'].fillna(0).values
grail_mean, grail_std = np.mean(grail_vec_tr), np.std(grail_vec_tr) + 1e-8

grail_z_te = ((test_df.loc[mask_safe, 'S03_A07_A05_V09_LagT1'].fillna(0).values + test_df.loc[mask_safe, 'Price_LagT3'].fillna(0).values) - grail_mean) / grail_std

m2_vec_tr = energy_product(relu(df_train_clean['S01_F03_U01_LagT2'].fillna(0).values), relu(df_train_clean['S02_F03_U01_LagT3'].fillna(0).values))
m2_mean, m2_std = np.mean(m2_vec_tr), np.std(m2_vec_tr) + 1e-8

m2_z_te = (energy_product(relu(test_df.loc[mask_safe, 'S01_F03_U01_LagT2'].fillna(0).values), relu(test_df.loc[mask_safe, 'S02_F03_U01_LagT3'].fillna(0).values)) - m2_mean) / m2_std

def calc_omega(df_subset):
    sq_d02_t2 = sign_sq(df_subset['S03_D02_V01_A01_B10_E10_E11_LagT2'].fillna(0).values)
    sq_d02_t3 = sign_sq(df_subset['S03_D02_V01_A01_B10_E10_E11_LagT3'].fillna(0).values)
    sq_a07_t2 = sign_sq(df_subset['S03_A07_V01_V09_LagT2'].fillna(0).values)
    relu_a07_t1 = relu(df_subset['S03_A07_A05_V09_LagT1'].fillna(0).values)
    
    block1 = sub(sq_d02_t2, sq_d02_t3)
    l2_a = max_pool(block1, interaction_abs(sq_a07_t2, sq_d02_t3))
    l2_b = max_pool(block1, interaction_abs(sq_d02_t2, sq_a07_t2))
    l2_c = sub(block1, interaction_abs(relu_a07_t1, sq_d02_t3))
    return max_pool(interaction_abs(l2_a, l2_b), max_pool(l2_a, l2_c))

m3_vec_tr = calc_omega(df_train_clean)
m3_mean, m3_std = np.mean(m3_vec_tr), np.std(m3_vec_tr) + 1e-8

m3_z_te = (calc_omega(test_df.loc[mask_safe]) - m3_mean) / m3_std

base_mod1_te = base_test_preds * (1.0 + (CFG.mod1_C * grail_z_te))
base_mod2_te = base_mod1_te * (1.0 + (CFG.mod2_C * m2_z_te))
adjusted_test_preds = base_mod2_te * (1.0 + (CFG.mod3_C * m3_z_te))

final_scaled_test_preds = np.copy(adjusted_test_preds)
pos_mask = final_scaled_test_preds > 0
final_scaled_test_preds[pos_mask] = final_scaled_test_preds[pos_mask] * CFG.asym_positive_scale

test_df['Backward_Estimate'] = 0.0 
test_df.loc[mask_safe, 'Backward_Estimate'] = final_scaled_test_preds

test_df['FINAL_SUBMISSION'] = np.clip(test_df['Backward_Estimate'].values, lower_bound_val, upper_bound_val)

submission = pd.DataFrame({'ID': test_df['ID'], 'TARGET': test_df['FINAL_SUBMISSION']})
submission.to_csv('submission.csv', index=False)

print("Execution Complete: submission.csv generated.")

Executing Production Pipeline...
Execution Complete: submission.csv generated.
